# Fig. 2: detector waveform and trigger levels

Reads `../data/decent.npy` (figshare item `waveform`) and writes `../figs/waveform_{light,dark}.{pdf,svg}` (manuscript Fig. 2) and `../data/tigger_level_time_difference.json` (the 1-photon t_B - t_A delay, 548.94 ps, hard-coded in the 20 GHz notebooks as `xy_delay_vector`).

Some rows of the scan are shifted/rescaled by hand (`np.roll`) to repair a time-tagger artefact, as in the published figure.


In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt
from snsphd import viz
import math
import orjson
Colors, pallet = viz.phd_style(jupyterStyle = True, grid = True, axese_width=0.5, text = 6)

from pydantic import BaseModel
from enum import Enum

from snsphd.layout import bisect

In [ ]:
fig = plt.figure(figsize=(10, 5)) # set the figure size to 10x5
# Load the .npy file
data = np.load('../data/decent.npy')
plt.imshow(data, cmap='gray', aspect='auto') # set the aspect ratio to 'auto'
plt.show()

In [ ]:
data_f = np.copy(data)

data_portion = data_f[170:200,:500]

# a manual correction for a defect from a weird timetagger bug 
# (these would appear randomly at different trigger levels. This was
# the best scan I got)
data_portion[11] = np.roll(data_portion[11], -5)*1.03
data_portion[12] = np.roll(data_portion[12], -15)*1.05
data_portion[13] = np.roll(data_portion[13], -20)*1.15
data_portion[14] = np.roll(data_portion[14], -13)*1.15
data_portion[15] = np.roll(data_portion[15], -8)*1.05
data_portion[16] = np.roll(data_portion[16], -3)*1.02

data_f[36] = np.roll(data_f[36], -3)*0.7


fig = plt.figure(figsize=(10, 5))
plt.imshow(data_f, cmap='viridis', aspect='auto') # set the aspect ratio to 'auto'
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.lines import Line2D


# \usepackage{amsmath}
# \usepackage[mathrm=sym]{unicode-math}
# \setmathfont{Fira Math}
# plt.rcParams['text.latex.preamble'] = r'\usepackage{amsmath}'
# plt.rcParams['text.latex.preamble'] = r'\usepackage[mathrm=sym]{unicode-math}'
# plt.rcParams['text.latex.preamble'] = r'\setmathfont{Fira Math}'
import shutil
# the panel labels use LaTeX (\bm) if a LaTeX installation is available, matplotlib mathtext otherwise
USETEX = shutil.which("latex") is not None
ket = (lambda n: rf"$\bm{{|{n}\rangle}}$") if USETEX else (lambda n: rf"$\mathbf{{|{n}\rangle}}$")
plt.rcParams['text.latex.preamble'] = r'\usepackage{bm}'



fig = plt.figure(figsize=(8, 8)) # set the figure size to 10x5


bottom, top_middle = bisect([0,0,1,1], direction='horiz', offset=.2, spacing = 0.07)
middle, top = bisect(top_middle, direction='horiz', offset=1-0.2/0.8, spacing = 0.07/0.8)

ax = fig.add_axes(middle) # main axes
ax_bottom = fig.add_axes(bottom) # inset axes
ax_top = fig.add_axes(top) # inset axes





def create_brighter_cmap():
    # create a scaled version of Viridis
    viridis = plt.get_cmap('viridis')
    newcolors = viridis(np.linspace(0, 1, 256))
    newcolors = np.mean([newcolors, newcolors], axis=0)
    newcmp = mcolors.LinearSegmentedColormap.from_list("Custom", newcolors)

    return newcmp


def voltage_to_idx_y(v):
    v_array = np.linspace(5, 60, np.shape(data_cropped)[0])
    idx = np.arange(0, np.shape(data_cropped)[0], 1)
    return int(np.interp(v, v_array, idx))


data_cropped = data_f[15:,50:1100]
data_cropped = np.flip(data_cropped, axis=0)


x = np.arange(0, np.shape(data_cropped)[1], 1)
y = np.linspace(5, 60, np.shape(data_cropped)[0])

voltage_to_idx_y(50)


ax.imshow(data_cropped, cmap=create_brighter_cmap(), aspect='auto', extent=[min(x), max(x), min(y), max(y)], origin="lower") # set the aspect ratio to 'auto'
ax.grid(False)


trigger_color = 'deeppink'
ax.axhline(y=8, color=trigger_color, linestyle='-', alpha=0.3)
ax.axhline(y=50, color=trigger_color, linestyle='-', alpha=0.3)


ax.plot([450, 900], [50,50], color=trigger_color, linestyle='-', alpha=1)
ax.plot([50, 300], [8,8], color=trigger_color, linestyle='-', alpha=1)

length = 0.8
ax.plot([450, 450], [50-length, 50+length], color=trigger_color, linestyle='-', alpha=1)
ax.plot([900, 900], [50-length, 50+length], color=trigger_color, linestyle='-', alpha=1)

ax.plot([50, 50], [8-length, 8+length], color=trigger_color, linestyle='-', alpha=1)
ax.plot([300, 300], [8-length, 8+length], color=trigger_color, linestyle='-', alpha=1)

ax.set_xlabel('Time (ps)')
ax.set_ylabel('Voltage (mV)')


def make_line(start_ax, start_coord, end_ax, end_coord, fig, **plot_kwargs):
    pt1_display = start_ax.transData.transform(start_coord)
    pt1_figure = fig.transSubfigure.inverted().transform(pt1_display)

    pt2_display = end_ax.transData.transform(end_coord)
    pt2_figure = fig.transSubfigure.inverted().transform(pt2_display)

    xpts = [pt1_figure[0], pt2_figure[0]]
    ypts = [pt1_figure[1], pt2_figure[1]]

    end_ax.plot(xpts, ypts, transform=fig.transFigure, clip_on=False, **plot_kwargs)



def make_zoom(line_start_coords, line_end_coords, ax, ax_inset, fig, loc = 'top', **plot_kwargs):
    if loc=='top':
        inset_coord_1 = (0,0)
        inset_coord_2 = (1,0)
    elif loc=='bottom':
        inset_coord_1 = (0,1)
        inset_coord_2 = (1,1)
    make_line(ax, line_start_coords, ax_inset, inset_coord_1, fig, **plot_kwargs, solid_capstyle='round')
    make_line(ax, line_end_coords, ax_inset, inset_coord_2, fig, **plot_kwargs, solid_capstyle='round')

make_zoom((450, 50), (900, 50), ax, ax_top, fig, loc='top', color='lightgray', linestyle='-', alpha=0.5)
make_zoom((50, 8), (300, 8), ax, ax_bottom, fig, loc='bottom', color='lightgray', linestyle='-', alpha=0.5)

make_line(ax_top, (1,0), ax_top, (1,1), fig, color='lightgray', linestyle='-', alpha=0.5)
make_line(ax_top, (0,1), ax_top, (1,1), fig, color='lightgray', linestyle='-', alpha=0.5)

make_line(ax_top, (0,0), ax_top, (0,1), fig, color='lightgray', linestyle='-', alpha=1, zorder=100)
make_line(ax_top, (0,0), ax_top, (1,0), fig, color='lightgray', linestyle='-', alpha=1, zorder=100)
# make_line(ax_top, (0,0), ax_top, (1,0), fig, color=trigger_color, linestyle='-', alpha=1, zorder=100)

# ax_top.plot((0,0), (-.05,.05), color=trigger_color, linestyle='-', transform=ax_top.transAxes, alpha=1, zorder=100, clip_on=False)
# ax_top.plot((1,1), (-.05,.05), color=trigger_color, linestyle='-', transform=ax_top.transAxes, alpha=1, zorder=100, clip_on=False)

make_line(ax_bottom, (1,0), ax_bottom, (1,1), fig, color='lightgray', linestyle='-', alpha=1)
make_line(ax_bottom, (0,1), ax_bottom, (1,1), fig, color='lightgray', linestyle='-', alpha=1)
make_line(ax_bottom, (0,0), ax_bottom, (0,1), fig, color='lightgray', linestyle='-', alpha=1, zorder=100)
make_line(ax_bottom, (0,0), ax_bottom, (1,0), fig, color='lightgray', linestyle='-', alpha=1, zorder=100)


ax_top.plot(np.arange(450, 900), data_cropped[voltage_to_idx_y(50),450:900], color='k', linestyle='-', alpha=1)
ax_top.set_xlabel("Time (ps)")
ax_top.set_ylabel("Counts")
ax_top.text(780, 300, ket('1'), fontsize=16, usetex=USETEX)
ax_top.text(620, 200, ket('2'), fontsize=16, usetex=USETEX)
ax_top.text(555, 110, ket('3'), fontsize=16, usetex=USETEX)




ax_bottom.plot(np.arange(50, 300), data_cropped[voltage_to_idx_y(8),50:300], color='k', linestyle='-', alpha=1)
ax_bottom.set_xlabel("Time (ps)")
ax_bottom.set_ylabel("Counts")

ax.text(600, 8 + 2, "Trigger A: 8 mV", transform=ax.transData, ha='center', va='center', color=trigger_color, fontsize=15, alpha=0.5)
ax.text(150, 50 + 2, "Trigger B: 50 mV", transform=ax.transData, ha='center', va='center', color=trigger_color, fontsize=15, alpha=0.5)


viz.save_light_dark_all(fig, name="../figs/waveform", override_alpha=True)

In [ ]:
len(data_cropped[0])

In [ ]:
plt.plot(data_cropped[voltage_to_idx_y(50),:])

In [ ]:
plt.plot(np.arange(1050), data_cropped[voltage_to_idx_y(8),:])

low_single_photon_time = np.sum(data_cropped[voltage_to_idx_y(8),188:400]*np.arange(188, 400))/np.sum(data_cropped[voltage_to_idx_y(8),188:400])

low_single_photon_time = np.average(np.arange(188, 400), weights=data_cropped[voltage_to_idx_y(8),188:400])

plt.plot(data_cropped[voltage_to_idx_y(8),188:])
plt.axvline(x=low_single_photon_time, color='r')


In [ ]:
plt.plot(np.arange(1050), data_cropped[voltage_to_idx_y(50),:])

high_single_photon_time = np.sum(data_cropped[voltage_to_idx_y(50),700:900]*np.arange(700, 900))/np.sum(data_cropped[voltage_to_idx_y(8),700:900])

high_single_photon_time = np.average(np.arange(700, 900), weights=data_cropped[voltage_to_idx_y(50),700:900])

# # plt.plot(data_cropped[voltage_to_idx_y(8),188:])

print(high_single_photon_time)
plt.axvline(x=high_single_photon_time, color='r')

In [ ]:
high_low_difference = high_single_photon_time - low_single_photon_time
high_low_difference

with open("../data/tigger_level_time_difference.json", "w") as file:
    file.write(json.dumps(high_low_difference))

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.lines import Line2D


# \usepackage{amsmath}
# \usepackage[mathrm=sym]{unicode-math}
# \setmathfont{Fira Math}
# plt.rcParams['text.latex.preamble'] = r'\usepackage{amsmath}'
# plt.rcParams['text.latex.preamble'] = r'\usepackage[mathrm=sym]{unicode-math}'
# plt.rcParams['text.latex.preamble'] = r'\setmathfont{Fira Math}'
import shutil
# the panel labels use LaTeX (\bm) if a LaTeX installation is available, matplotlib mathtext otherwise
USETEX = shutil.which("latex") is not None
ket = (lambda n: rf"$\bm{{|{n}\rangle}}$") if USETEX else (lambda n: rf"$\mathbf{{|{n}\rangle}}$")
plt.rcParams['text.latex.preamble'] = r'\usepackage{bm}'



fig = plt.figure(figsize=(3.5, 5)) # set the figure size to 10x5


# bottom, top_middle = bisect([0,0,1,1], direction='horiz', offset=.2, spacing = 0.07)
# middle, top = bisect(top_middle, direction='horiz', offset=1-0.2/0.8, spacing = 0.07/0.8)
middle = [0,0,1,1]

ax = fig.add_axes(middle) # main axes
# ax_bottom = fig.add_axes(bottom) # inset axes
# ax_top = fig.add_axes(top) # inset axes





def create_brighter_cmap():
    # create a scaled version of Viridis
    viridis = plt.get_cmap('viridis')
    newcolors = viridis(np.linspace(0, 1, 256))
    newcolors = np.mean([newcolors, newcolors], axis=0)
    newcmp = mcolors.LinearSegmentedColormap.from_list("Custom", newcolors)

    return newcmp


def voltage_to_idx_y(v):
    v_array = np.linspace(5, 60, np.shape(data_cropped)[0])
    idx = np.arange(0, np.shape(data_cropped)[0], 1)
    return int(np.interp(v, v_array, idx))


data_cropped = data_f[15:,50:1100]
data_cropped = np.flip(data_cropped, axis=0)


x = np.arange(0, np.shape(data_cropped)[1], 1)
y = np.linspace(5, 60, np.shape(data_cropped)[0])

voltage_to_idx_y(50)


ax.imshow(data_cropped, cmap=create_brighter_cmap(), aspect='auto', extent=[min(x), max(x), min(y), max(y)], origin="lower") # set the aspect ratio to 'auto'
ax.grid(True)


trigger_color = 'deeppink'
# ax.axhline(y=8, color=trigger_color, linestyle='-', alpha=0.3)
# ax.axhline(y=50, color=trigger_color, linestyle='-', alpha=0.3)


# ax.plot([450, 900], [50,50], color=trigger_color, linestyle='-', alpha=1)
# ax.plot([50, 300], [8,8], color=trigger_color, linestyle='-', alpha=1)

# length = 0.8
# ax.plot([450, 450], [50-length, 50+length], color=trigger_color, linestyle='-', alpha=1)
# ax.plot([900, 900], [50-length, 50+length], color=trigger_color, linestyle='-', alpha=1)

# ax.plot([50, 50], [8-length, 8+length], color=trigger_color, linestyle='-', alpha=1)
# ax.plot([300, 300], [8-length, 8+length], color=trigger_color, linestyle='-', alpha=1)

ax.set_xlabel('Time (ps)')
ax.set_ylabel('Voltage (mV)')


def make_line(start_ax, start_coord, end_ax, end_coord, fig, **plot_kwargs):
    pt1_display = start_ax.transData.transform(start_coord)
    pt1_figure = fig.transSubfigure.inverted().transform(pt1_display)

    pt2_display = end_ax.transData.transform(end_coord)
    pt2_figure = fig.transSubfigure.inverted().transform(pt2_display)

    xpts = [pt1_figure[0], pt2_figure[0]]
    ypts = [pt1_figure[1], pt2_figure[1]]

    end_ax.plot(xpts, ypts, transform=fig.transFigure, clip_on=False, **plot_kwargs)



def make_zoom(line_start_coords, line_end_coords, ax, ax_inset, fig, loc = 'top', **plot_kwargs):
    if loc=='top':
        inset_coord_1 = (0,0)
        inset_coord_2 = (1,0)
    elif loc=='bottom':
        inset_coord_1 = (0,1)
        inset_coord_2 = (1,1)
    make_line(ax, line_start_coords, ax_inset, inset_coord_1, fig, **plot_kwargs, solid_capstyle='round')
    make_line(ax, line_end_coords, ax_inset, inset_coord_2, fig, **plot_kwargs, solid_capstyle='round')

# make_zoom((450, 50), (900, 50), ax, ax_top, fig, loc='top', color='lightgray', linestyle='-', alpha=0.5)
# make_zoom((50, 8), (300, 8), ax, ax_bottom, fig, loc='bottom', color='lightgray', linestyle='-', alpha=0.5)

# make_line(ax_top, (1,0), ax_top, (1,1), fig, color='lightgray', linestyle='-', alpha=0.5)
# make_line(ax_top, (0,1), ax_top, (1,1), fig, color='lightgray', linestyle='-', alpha=0.5)

# make_line(ax_top, (0,0), ax_top, (0,1), fig, color='lightgray', linestyle='-', alpha=1, zorder=100)
# make_line(ax_top, (0,0), ax_top, (1,0), fig, color='lightgray', linestyle='-', alpha=1, zorder=100)
# # make_line(ax_top, (0,0), ax_top, (1,0), fig, color=trigger_color, linestyle='-', alpha=1, zorder=100)

# # ax_top.plot((0,0), (-.05,.05), color=trigger_color, linestyle='-', transform=ax_top.transAxes, alpha=1, zorder=100, clip_on=False)
# # ax_top.plot((1,1), (-.05,.05), color=trigger_color, linestyle='-', transform=ax_top.transAxes, alpha=1, zorder=100, clip_on=False)

# make_line(ax_bottom, (1,0), ax_bottom, (1,1), fig, color='lightgray', linestyle='-', alpha=1)
# make_line(ax_bottom, (0,1), ax_bottom, (1,1), fig, color='lightgray', linestyle='-', alpha=1)
# make_line(ax_bottom, (0,0), ax_bottom, (0,1), fig, color='lightgray', linestyle='-', alpha=1, zorder=100)
# make_line(ax_bottom, (0,0), ax_bottom, (1,0), fig, color='lightgray', linestyle='-', alpha=1, zorder=100)


# ax_top.plot(np.arange(450, 900), data_cropped[voltage_to_idx_y(50),450:900], color='k', linestyle='-', alpha=1)
# ax_top.set_xlabel("Time (ps)")
# ax_top.set_ylabel("Counts")
# ax_top.text(780, 300, ket('1'), fontsize=16, usetex=USETEX)
# ax_top.text(620, 200, ket('2'), fontsize=16, usetex=USETEX)
# ax_top.text(555, 110, ket('3'), fontsize=16, usetex=USETEX)




# ax_bottom.plot(np.arange(50, 300), data_cropped[voltage_to_idx_y(8),50:300], color='k', linestyle='-', alpha=1)
# ax_bottom.set_xlabel("Time (ps)")
# ax_bottom.set_ylabel("Counts")

# ax.text(600, 8 + 2, "Trigger A: 8 mV", transform=ax.transData, ha='center', va='center', color=trigger_color, fontsize=15, alpha=0.5)
# ax.text(150, 50 + 2, "Trigger B: 50 mV", transform=ax.transData, ha='center', va='center', color=trigger_color, fontsize=15, alpha=0.5)


viz.save_light_dark_all(fig, name="../figs/waveform_simple", override_alpha=True)

In [ ]:
# Ordinary function that just returns a list
def record_breaking_values(inputs):
    it = iter(inputs)
    print(it)
    best = next(it)
    print(best)
    records = [best]
    for value in it:
        if value > best:
            best = value
            records.append(best)
    return records

In [ ]:
record_breaking_values([4,3,45,65,3,3])

In [ ]:
# Written as a generator
def record_breaking_values(inputs):
    it = iter(inputs)
    best = next(it)
    yield best
    for value in it:
        if value > best:
            best = value
            yield best

In [ ]:
for item in record_breaking_values([4,3,45,65,3,3]):
    print(item)

In [ ]:
# import plotly.graph_objects as go
# import numpy as np

# # Helix equation
# t = np.linspace(0, 10, 50)
# x, y, z = np.cos(t), np.sin(t), t

# fig= go.Figure(go.Scatter3d(x=x, y=y, z=z, mode='markers'))

# x_eye = -1.25
# y_eye = 2
# z_eye = 0.5

# fig.update_layout(
#          title='Animation Test',
#          width=600,
#          height=600,
#          scene_camera_eye=dict(x=x_eye, y=y_eye, z=z_eye),
#          updatemenus=[dict(type='buttons',
#                   showactive=False,
#                   y=1,
#                   x=0.8,
#                   xanchor='left',
#                   yanchor='bottom',
#                   pad=dict(t=45, r=10),
#                   buttons=[dict(label='Play',
#                                  method='animate',
#                                  args=[None, dict(frame=dict(duration=5, redraw=True), 
#                                                              transition=dict(duration=0),
#                                                              fromcurrent=True,
#                                                              mode='immediate'
#                                                             )]
#                                             )
#                                       ]
#                               )
#                         ]
# )

# fig.update_geos(projection_type="equirectangular", visible=True, resolution=50)


# frames=[]
# for t in np.arange(0, 6.26, 0.01):
#     xe, ye, ze = rotate_z(x_eye, y_eye, z_eye, -t)
#     frames.append(go.Frame(layout=dict(scene_camera_eye=dict(x=xe, y=ye, z=ze))))
# fig.frames=frames

# fig.show()

In [ ]:
# import numpy as np
# import plotly.graph_objects as go

# # Generate nicely looking random 3D-field
# np.random.seed(0)
# l = 30
# X, Y, Z = np.mgrid[:l, :l, :l]
# vol = np.zeros((l, l, l))
# pts = (l * np.random.rand(3, 15)).astype(np.int64)
# vol[tuple(indices for indices in pts)] = 1
# from scipy import ndimage
# vol = ndimage.gaussian_filter(vol, 4)
# vol /= vol.max()

# def rotate_z(x, y, z, theta):
#     w = x+1j*y
#     return np.real(np.exp(1j*theta)*w), np.imag(np.exp(1j*theta)*w), z

# x_eye = -1.25
# y_eye = 2
# z_eye = 0.5

# fig= go.Figure(go.Volume(
#             x=X.flatten(), y=Y.flatten(), z=Z.flatten(),
#             value=vol.flatten(),
#             isomin=0.2,
#             isomax=0.7,
#             opacity=0.1,
#             surface_count=25,
#             ))
    

# fig.update_layout(
#          title='Animation Test',
#          width=600,
#          height=600,
#          scene_camera_eye=dict(x=x_eye, y=y_eye, z=z_eye),
#          updatemenus=[dict(type='buttons',
#                   showactive=False,
#                   y=1,
#                   x=0.8,
#                   xanchor='left',
#                   yanchor='bottom',
#                   pad=dict(t=45, r=10),
#                   buttons=[dict(label='Play',
#                                  method='animate',
#                                  args=[None, dict(frame=dict(duration=5, redraw=True), 
#                                                              transition=dict(duration=0),
#                                                              fromcurrent=True,
#                                                              mode='after'
#                                                             )]
#                                             )
#                                       ]
#                               )
#                         ]
# )

# fig.update_geos(projection_type="equirectangular", visible=True, resolution=50)


# frames=[]
# for t in np.arange(0, 6.26, 0.01):
#     xe, ye, ze = rotate_z(x_eye, y_eye, z_eye, -t)
#     frames.append(go.Frame(layout=dict(scene_camera_eye=dict(x=xe, y=ye, z=ze))))
# fig.frames=frames

# fig.show()
# # fig.update_layout(scene_xaxis_showticklabels=False,
# #                   scene_yaxis_showticklabels=False,
# #                   scene_zaxis_showticklabels=False)
# fig.show()

In [ ]:
import numpy as np
import plotly.graph_objects as go

# Generate nicely looking random 3D-field
np.random.seed(0)
l = 30
X, Y, Z = np.mgrid[:l, :l, :l]
vol = np.zeros((l, l, l))
pts = (l * np.random.rand(3, 15)).astype(np.int64)
vol[tuple(indices for indices in pts)] = 1
from scipy import ndimage
vol = ndimage.gaussian_filter(vol, 4)
vol /= vol.max()

fig = go.Figure(data=go.Volume(
    x=X.flatten(), y=Y.flatten(), z=Z.flatten(),
    value=vol.flatten(),
    isomin=0.2,
    isomax=0.7,
    opacity=0.1,
    surface_count=25,
    ))
fig.update_layout(scene_xaxis_showticklabels=False,
                  scene_yaxis_showticklabels=False,
                  scene_zaxis_showticklabels=False)
fig.show()